# Phase 2c · Count publications and calculate the PII

After the pathogen–host screen, query PubMed for total pathogen publications only for pathogen IDs accepted in at least one selected host. The PII uses this total publication count, not an extrapolation from the sampled yes/no fraction. It is `log1p(total PubMed records)` normalized to 0–1 within pathogen group.

## Step 1 · Load accepted pathogen–host pairs

In [ ]:
import hashlib, json, os, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
from pathogen_atlas_v4.runs import resolve_run
from pathogen_atlas_v4.pubmed_client import count_accepted_pathogen_publications
from pathogen_atlas_v4.interest_index import build_pathogenic_interest_index
CONFIG = yaml.safe_load((PROJECT_ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('02c_pathogenic_interest_index', RUN_ID)
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
PATHOGENS = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
ALIASES = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_aliases.parquet', columns=['pathogen_id','name'])
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)), known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)), host_group_ids={k:g.host_id.astype(str).tolist() for k,g in HOSTS.groupby('host_group')}, pathogen_group_ids={k:g.pathogen_id.astype(str).tolist() for k,g in PATHOGENS.groupby('pathogen_group')})
cache_identity = {'pathogen_ids': sorted(resolved.pathogen_ids), 'query_version': CONFIG['acquisition']['query_strategy_version'], 'pathogen_literature_query_version': CONFIG['acquisition']['pathogen_literature_query_version'], 'taxonomy_snapshot': TAXONOMY_ROOT.parent.name}
cache_key = hashlib.sha256(json.dumps(cache_identity, sort_keys=True, default=list).encode()).hexdigest()
CACHE_DIR = ROOT / 'data/v4/runs' / RUN_ID / 'literature' / cache_key
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
PAIR_SELECTION_PATH = RUN_DIR / 'pathogen_host_selection.parquet'
COUNT_PATH = CACHE_DIR / 'tables/accepted_pathogen_publication_counts.parquet'
PII_PATH = RUN_DIR / 'release/pathogenic_interest_index.parquet'
if not PAIR_SELECTION_PATH.exists():
    raise FileNotFoundError(f'Run 02b and complete pathogen screening first: {PAIR_SELECTION_PATH}')
pair_selection = pd.read_parquet(PAIR_SELECTION_PATH)
accepted_pairs = pair_selection.loc[pair_selection.selection_status.eq('accepted') & pair_selection.selected_as_pathogen_for_host.fillna(False).astype(bool)].copy()
accepted_pathogen_ids = sorted(accepted_pairs.pathogen_id.astype(str).unique())
print('Accepted pathogen–host pairs:', len(accepted_pairs), '| unique accepted pathogen IDs:', len(accepted_pathogen_ids))
print('Only these pathogen IDs are sent to the PubMed total-publication count query.')
LOG.info('PII preflight: accepted_pairs=%d accepted_pathogens=%d', len(accepted_pairs), len(accepted_pathogen_ids))

## Step 2 · Query total publications for accepted pathogens only

In [ ]:
RUN_PUBMED_COUNT = False  # Set True to run/resume NCBI count-only searches for accepted pathogen IDs.
if RUN_PUBMED_COUNT:
    NCBI_EMAIL = os.environ.get('NCBI_EMAIL') or input('NCBI contact email: ').strip()
    NCBI_API_KEY = os.environ.get('NCBI_API_KEY') or None
    counts = count_accepted_pathogen_publications(
        accepted_pathogen_ids, PATHOGENS, ALIASES, CACHE_DIR,
        query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
        email=NCBI_EMAIL, api_key=NCBI_API_KEY, logger=LOG)
    display(counts.head(20))
else:
    counts = pd.read_parquet(COUNT_PATH) if COUNT_PATH.exists() else pd.DataFrame()
    print('PubMed count queries skipped. Set RUN_PUBMED_COUNT=True to search only the accepted pathogen set.')
    print('Cached accepted-pathogen counts available:', len(counts))

## Step 3 · Calculate and save the Pathogenic Interest Index

In [ ]:
if not accepted_pathogen_ids:
    pii = pd.DataFrame(columns=['pathogen_id','accepted_name','pathogen_group','n_pubmed_search_records','log_publication_count','pathogenic_interest_index','index_version','query_hash','query_version'])
    print('No pathogen–host pairs passed the evidence rule; no PII rows are emitted.')
elif counts.empty:
    raise FileNotFoundError(f'Run accepted-pathogen PubMed count queries first: {COUNT_PATH}')
else:
    pii = build_pathogenic_interest_index(counts, pair_selection, PATHOGENS)
RUN_DIR.joinpath('release').mkdir(parents=True, exist_ok=True)
pii.to_parquet(PII_PATH, index=False)
print('Saved:', PII_PATH)
display(pii.sort_values('pathogenic_interest_index', ascending=False).head(25))
LOG.info('PII table saved: accepted_pathogens=%d rows=%d path=%s', len(accepted_pathogen_ids), len(pii), PII_PATH)

## Interpretation

A host-specific pair is accepted when at least one screened abstract explicitly supports disease caused by that pathogen in that host, with a supporting quote from the abstract. The organism’s total PubMed count is queried once for each unique pathogen accepted in one or more selected hosts. The PII is a pathogen-level publication-volume score; the accepted pair table determines which hosts display that pathogen. The 200-record sample is used for screening only and is not multiplied by a sample positivity fraction.